In [ ]:
!nvidia-smi

Thu Nov  5 07:07:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpumq1miv_".


In [ ]:
!apt-get install -y libsqlite3-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsqlite3-dev is already the newest version (3.45.1-1ubuntu2.8).
libsqlite3-dev set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "PS_20174392719_1491204439457_log.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "ealaxi/paysim1",
    file_path,
)

print("Rows:", len(df))
print(df.head())

# Save for the C++ program. index=False is important: without it,
# pandas adds an extra first column and the C++ parser reads the wrong fields.
df.to_csv("paysim.csv", index=False)

/tmp/ipykernel_443/2147924980.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'paysim1' dataset.
Rows: 6362620
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [ ]:
!head -3 paysim.csv
!wc -l paysim.csv

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
6362621 paysim.csv


In [ ]:
%%writefile db_ops.cu

#include <cstdio>
#include <cstdlib>
#include <climits>
#include <ctime>
#include <cmath>
#include <vector>
#include <string>
#include <algorithm>
#include <fstream>
#include <functional>
#include <sqlite3.h>
using namespace std;

#define THREADS 256
#define GBLOCKS 1024
#define NCAT 5
#define RUNS 5
typedef unsigned long long ull;

const char *typeNames[NCAT] = {"CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"};
const double INDEX_LIMIT = 0.5;
double lastFrac = 0;

// ------------------------------------------------------------
//  Global data
// ------------------------------------------------------------
int n = 0, blocks = 0;

vector<int> h_cat, h_amt, h_flag;
vector<int> s_amt;

int *d_cat, *d_amt;
int *d_scat, *d_samt;
int *d_flag, *d_pInt;
long long *d_pLL;
ull *d_gSum, *d_gCnt;
vector<int> hInt;
vector<long long> hLL;


int *p_cat, *p_amt;
int m = 0, mblocks = 0;

void setFullScan() {
    p_cat = d_cat;  p_amt = d_amt;
    m = n;          mblocks = blocks;
}

// HYBRID STEP + OPTIMIZER (runs on the CPU):

bool planScan(int T) {
    int lo = (int)(upper_bound(s_amt.begin(), s_amt.end(), T) - s_amt.begin());
    int rows = n - lo;
    lastFrac = (double)rows / n;
    if (rows > INDEX_LIMIT * n) { setFullScan(); return false; }
    p_cat = d_scat + lo;  p_amt = d_samt + lo;
    m = rows;
    mblocks = max(1, (m + THREADS - 1) / THREADS);
    return true;
}

// ============================================================
//  SEQUENTIAL OPERATORS
// ============================================================


void cpuSelection(int C, int T) {
    for (int i = 0; i < n; i++)
        h_flag[i] = ((C < 0 || h_cat[i] == C) && h_amt[i] > T) ? 1 : 0;
}
long long cpuCount() {
    long long c = 0;
    for (int i = 0; i < n; i++) c += h_flag[i];
    return c;
}
long long cpuSum() {
    long long s = 0;
    for (int i = 0; i < n; i++) if (h_flag[i]) s += h_amt[i];
    return s;
}
int cpuMin() {
    int mn = INT_MAX;
    for (int i = 0; i < n; i++) if (h_flag[i] && h_amt[i] < mn) mn = h_amt[i];
    return mn;
}
int cpuMax() {
    int mx = INT_MIN;
    for (int i = 0; i < n; i++) if (h_flag[i] && h_amt[i] > mx) mx = h_amt[i];
    return mx;
}

void cpuGroupBy(ull *sum, ull *cnt) {
    for (int k = 0; k < NCAT; k++) { sum[k] = 0; cnt[k] = 0; }
    for (int i = 0; i < n; i++)
        if (h_flag[i]) { sum[h_cat[i]] += h_amt[i]; cnt[h_cat[i]]++; }
}

// ============================================================
//  CUDA KERNELS
// ============================================================

// SELECTION: each thread checks ONE row
__global__ void selectionKernel(int *cat, int *amt, int *flag, int n, int C, int T) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        flag[i] = ((C < 0 || cat[i] == C) && amt[i] > T) ? 1 : 0;
}

// The four aggregation kernels use a "tree reduction": each thread puts one
// value in shared memory (fast memory shared by the block), then values are
// combined in halves: 256 -> 128 -> ... -> 1. Each block writes ONE partial
// result and the CPU adds the partials.

__global__ void countKernel(int *flag, int n, int *partial) {
    __shared__ int s[THREADS];
    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;
    s[tid] = (i < n) ? flag[i] : 0;
    __syncthreads();
    for (int step = blockDim.x / 2; step > 0; step >>= 1) {
        if (tid < step) s[tid] += s[tid + step];
        __syncthreads();
    }
    if (tid == 0) partial[blockIdx.x] = s[0];
}

__global__ void sumKernel(int *amt, int *flag, int n, long long *partial) {
    __shared__ long long s[THREADS];
    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;
    s[tid] = (i < n && flag[i]) ? amt[i] : 0;
    __syncthreads();
    for (int step = blockDim.x / 2; step > 0; step >>= 1) {
        if (tid < step) s[tid] += s[tid + step];
        __syncthreads();
    }
    if (tid == 0) partial[blockIdx.x] = s[0];
}

__global__ void minKernel(int *amt, int *flag, int n, int *partial) {
    __shared__ int s[THREADS];
    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;
    s[tid] = (i < n && flag[i]) ? amt[i] : INT_MAX;
    __syncthreads();
    for (int step = blockDim.x / 2; step > 0; step >>= 1) {
        if (tid < step && s[tid + step] < s[tid]) s[tid] = s[tid + step];
        __syncthreads();
    }
    if (tid == 0) partial[blockIdx.x] = s[0];
}

__global__ void maxKernel(int *amt, int *flag, int n, int *partial) {
    __shared__ int s[THREADS];
    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;
    s[tid] = (i < n && flag[i]) ? amt[i] : INT_MIN;
    __syncthreads();
    for (int step = blockDim.x / 2; step > 0; step >>= 1) {
        if (tid < step && s[tid + step] > s[tid]) s[tid] = s[tid + step];
        __syncthreads();
    }
    if (tid == 0) partial[blockIdx.x] = s[0];
}


__device__ ull warpAdd(ull v) {
    for (int d = 16; d > 0; d >>= 1) v += __shfl_down_sync(0xffffffff, v, d);
    return v;
}


__global__ void groupKernel(int *cat, int *amt, int *flag, int n, ull *gSum, ull *gCnt) {
    ull s[NCAT], c[NCAT];                                // private counters
    #pragma unroll
    for (int k = 0; k < NCAT; k++) { s[k] = 0; c[k] = 0; }

    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n;
         i += gridDim.x * blockDim.x) {                  // jump by the total thread count
        if (flag[i]) {
            #pragma unroll
            for (int k = 0; k < NCAT; k++)
                if (cat[i] == k) { s[k] += amt[i]; c[k]++; }
        }
    }

    #pragma unroll
    for (int k = 0; k < NCAT; k++) { s[k] = warpAdd(s[k]); c[k] = warpAdd(c[k]); }
    if ((threadIdx.x & 31) == 0) {
        #pragma unroll
        for (int k = 0; k < NCAT; k++) { atomicAdd(&gSum[k], s[k]); atomicAdd(&gCnt[k], c[k]); }
    }
}

// ============================================================
//  CUDA OPERATORS:
// ============================================================
void gpuSelection(int C, int T) {
    selectionKernel<<<mblocks, THREADS>>>(p_cat, p_amt, d_flag, m, C, T);
    cudaDeviceSynchronize();
}
long long gpuCount() {
    countKernel<<<mblocks, THREADS>>>(d_flag, m, d_pInt);
    cudaDeviceSynchronize();
    cudaMemcpy(hInt.data(), d_pInt, mblocks * sizeof(int), cudaMemcpyDeviceToHost);
    long long c = 0;
    for (int b = 0; b < mblocks; b++) c += hInt[b];
    return c;
}
long long gpuSum() {
    sumKernel<<<mblocks, THREADS>>>(p_amt, d_flag, m, d_pLL);
    cudaDeviceSynchronize();
    cudaMemcpy(hLL.data(), d_pLL, mblocks * sizeof(long long), cudaMemcpyDeviceToHost);
    long long s = 0;
    for (int b = 0; b < mblocks; b++) s += hLL[b];
    return s;
}
int gpuMin() {
    minKernel<<<mblocks, THREADS>>>(p_amt, d_flag, m, d_pInt);
    cudaDeviceSynchronize();
    cudaMemcpy(hInt.data(), d_pInt, mblocks * sizeof(int), cudaMemcpyDeviceToHost);
    int mn = INT_MAX;
    for (int b = 0; b < mblocks; b++) if (hInt[b] < mn) mn = hInt[b];
    return mn;
}
int gpuMax() {
    maxKernel<<<mblocks, THREADS>>>(p_amt, d_flag, m, d_pInt);
    cudaDeviceSynchronize();
    cudaMemcpy(hInt.data(), d_pInt, mblocks * sizeof(int), cudaMemcpyDeviceToHost);
    int mx = INT_MIN;
    for (int b = 0; b < mblocks; b++) if (hInt[b] > mx) mx = hInt[b];
    return mx;
}
void gpuGroupBy(ull *sum, ull *cnt) {
    cudaMemset(d_gSum, 0, NCAT * sizeof(ull));
    cudaMemset(d_gCnt, 0, NCAT * sizeof(ull));
    groupKernel<<<min(mblocks, GBLOCKS), THREADS>>>(p_cat, p_amt, d_flag, m, d_gSum, d_gCnt);
    cudaDeviceSynchronize();
    cudaMemcpy(sum, d_gSum, NCAT * sizeof(ull), cudaMemcpyDeviceToHost);
    cudaMemcpy(cnt, d_gCnt, NCAT * sizeof(ull), cudaMemcpyDeviceToHost);
}

// ============================================================
//  GPU OPERATORS AS SQL FUNCTIONS
// ============================================================

void gpuAggSQL(sqlite3_context *ctx, int argc, sqlite3_value **argv) {
    string f = (const char *)sqlite3_value_text(argv[0]);
    int C = sqlite3_value_int(argv[1]);
    int T = sqlite3_value_int(argv[2]);

    planScan(T);                                            // CPU: optimizer
    gpuSelection(C, T);                                     // GPU SELECTION

    if (f == "count")      sqlite3_result_double(ctx, (double)gpuCount());
    else if (f == "sum")   sqlite3_result_double(ctx, (double)gpuSum());
    else if (f == "min")   { int v = gpuMin(); if (v == INT_MAX) sqlite3_result_null(ctx); else sqlite3_result_double(ctx, v); }
    else if (f == "max")   { int v = gpuMax(); if (v == INT_MIN) sqlite3_result_null(ctx); else sqlite3_result_double(ctx, v); }
    else sqlite3_result_error(ctx, "gpu_agg: use count, sum, min or max", -1);
}

struct GroupResult { ull sum[NCAT], cnt[NCAT]; };

void gpuGroupSQL(sqlite3_context *ctx, int argc, sqlite3_value **argv) {
    string f = (const char *)sqlite3_value_text(argv[0]);
    int cat  = sqlite3_value_int(argv[1]);
    int T    = sqlite3_value_int(argv[2]);

    GroupResult r;
    GroupResult *saved = (GroupResult *)sqlite3_get_auxdata(ctx, 2);
    if (saved) {
        r = *saved;
    } else {
        planScan(T);
        gpuSelection(-1, T);
        gpuGroupBy(r.sum, r.cnt);
        sqlite3_set_auxdata(ctx, 2, new GroupResult(r),
                            [](void *p) { delete (GroupResult *)p; });
    }
    if (cat < 0 || cat >= NCAT) { sqlite3_result_null(ctx); return; }
    sqlite3_result_double(ctx, f == "count" ? (double)r.cnt[cat] : (double)r.sum[cat]);
}


double timeIt(function<void()> f) {
    f();
    clock_t start = clock();
    for (int r = 0; r < RUNS; r++) f();
    return (double)(clock() - start) / CLOCKS_PER_SEC * 1000.0 / RUNS;
}


int typeCode(const string &s) {
    for (int k = 0; k < NCAT; k++) if (s == typeNames[k]) return k;
    return -1;
}


vector<pair<int, double>> runSQL(sqlite3 *db, const string &sql) {
    vector<pair<int, double>> rows;
    sqlite3_stmt *st;
    sqlite3_prepare_v2(db, sql.c_str(), -1, &st, 0);
    while (sqlite3_step(st) == SQLITE_ROW) {
        if (sqlite3_column_count(st) == 1) rows.push_back({0, sqlite3_column_double(st, 0)});
        else rows.push_back({sqlite3_column_int(st, 0), sqlite3_column_double(st, 1)});
    }
    sqlite3_finalize(st);
    return rows;
}


bool sameRows(const vector<pair<int, double>> &a, const vector<pair<int, double>> &b) {
    if (a.size() != b.size() || a.empty()) return false;
    for (size_t i = 0; i < a.size(); i++)
        if (a[i].first != b[i].first || fabs(a[i].second - b[i].second) > 0.01) return false;
    return true;
}


int main(int argc, char **argv) {
    const char *csvPath = (argc > 1) ? argv[1] : "paysim.csv";
    const int C  = 4;         // type = TRANSFER
    const int T  = 200000;    // amount > 200000   (selection and aggregates)
    const int TG = 10000;     // amount > 10000    (group by)

    // ---------- 1. Load the PaySim CSV into SQLite ----------
    ifstream file(csvPath);
    if (!file) { printf("Cannot open '%s'.\n", csvPath); return 1; }

    sqlite3 *db;
    sqlite3_open(":memory:", &db);                       // database kept in RAM
    sqlite3_exec(db, "CREATE TABLE transactions(type INT, amount INT);", 0, 0, 0);
    sqlite3_exec(db, "CREATE TABLE categories(id INT);", 0, 0, 0);
    for (int k = 0; k < NCAT; k++)
        sqlite3_exec(db, ("INSERT INTO categories VALUES(" + to_string(k) + ");").c_str(), 0, 0, 0);

    sqlite3_exec(db, "BEGIN;", 0, 0, 0);
    sqlite3_stmt *ins;
    sqlite3_prepare_v2(db, "INSERT INTO transactions VALUES(?, ?);", -1, &ins, 0);
    string line;
    getline(file, line);
    while (getline(file, line)) {
        size_t a = line.find(',');
        size_t b = line.find(',', a + 1);
        size_t c = line.find(',', b + 1);
        if (c == string::npos) continue;
        int code = typeCode(line.substr(a + 1, b - a - 1));
        int amt  = (int)(atof(line.substr(b + 1, c - b - 1).c_str()) + 0.5);
        if (code < 0) continue;
        sqlite3_bind_int(ins, 1, code);
        sqlite3_bind_int(ins, 2, amt);
        sqlite3_step(ins);
        sqlite3_reset(ins);
    }
    sqlite3_finalize(ins);
    sqlite3_exec(db, "COMMIT;", 0, 0, 0);

    // ---------- 2. Read the table out of SQLite into arrays ----------
    clock_t t1 = clock();
    sqlite3_stmt *rd;
    sqlite3_prepare_v2(db, "SELECT type, amount FROM transactions;", -1, &rd, 0);
    while (sqlite3_step(rd) == SQLITE_ROW) {
        h_cat.push_back(sqlite3_column_int(rd, 0));
        h_amt.push_back(sqlite3_column_int(rd, 1));
    }
    sqlite3_finalize(rd);
    double loadTime = (double)(clock() - t1) / CLOCKS_PER_SEC * 1000.0;

    n = (int)h_cat.size();
    blocks = (n + THREADS - 1) / THREADS;
    h_flag.resize(n);  hInt.resize(blocks);  hLL.resize(blocks);
    printf("PaySim loaded into SQLite: %d transactions\n", n);

    // ---------- 3. Copy the table to the GPU ----------
    cudaMalloc(&d_cat,  n * sizeof(int));   cudaMalloc(&d_amt,  n * sizeof(int));
    cudaMalloc(&d_flag, n * sizeof(int));
    cudaMalloc(&d_pInt, blocks * sizeof(int));
    cudaMalloc(&d_pLL,  blocks * sizeof(long long));
    cudaMalloc(&d_gSum, NCAT * sizeof(ull));
    cudaMalloc(&d_gCnt, NCAT * sizeof(ull));

    clock_t t2 = clock();
    cudaMemcpy(d_cat, h_cat.data(), n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_amt, h_amt.data(), n * sizeof(int), cudaMemcpyHostToDevice);
    double copyTime = (double)(clock() - t2) / CLOCKS_PER_SEC * 1000.0;

    // ---------- 4. HYBRID ----------

    clock_t t3 = clock();
    vector<ull> keys(n);
    for (int i = 0; i < n; i++) keys[i] = ((ull)h_amt[i] << 8) | (ull)h_cat[i];
    sort(keys.begin(), keys.end());
    vector<int> s_cat(n);
    s_amt.resize(n);
    for (int i = 0; i < n; i++) { s_amt[i] = (int)(keys[i] >> 8); s_cat[i] = (int)(keys[i] & 0xFF); }
    cudaMalloc(&d_scat, n * sizeof(int));   cudaMalloc(&d_samt, n * sizeof(int));
    cudaMemcpy(d_scat, s_cat.data(), n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_samt, s_amt.data(), n * sizeof(int), cudaMemcpyHostToDevice);
    double indexTime = (double)(clock() - t3) / CLOCKS_PER_SEC * 1000.0;

    // ================= PART A: CPU vs CUDA vs HYBRID =================
    double cpuT[6], gpuT[6], hybT[6];
    long long cCnt, cSum, gCnt, gSum, hCnt, hSum;
    int cMin, cMax, gMin, gMax, hMin, hMax;
    ull cS[NCAT], cC[NCAT], gS[NCAT], gC[NCAT], hS[NCAT], hC[NCAT];

    // --- Sequential CPU ---
    cpuT[0] = timeIt([&] { cpuSelection(C, T); });
    cpuT[1] = timeIt([&] { cCnt = cpuCount(); });
    cpuT[2] = timeIt([&] { cSum = cpuSum(); });
    cpuT[3] = timeIt([&] { cMin = cpuMin(); });
    cpuT[4] = timeIt([&] { cMax = cpuMax(); });
    cpuSelection(-1, TG);
    cpuT[5] = timeIt([&] { cpuGroupBy(cS, cC); });


    setFullScan();
    gpuT[0] = timeIt([&] { gpuSelection(C, T); });
    gpuT[1] = timeIt([&] { gCnt = gpuCount(); });
    gpuT[2] = timeIt([&] { gSum = gpuSum(); });
    gpuT[3] = timeIt([&] { gMin = gpuMin(); });
    gpuT[4] = timeIt([&] { gMax = gpuMax(); });
    gpuSelection(-1, TG);
    gpuT[5] = timeIt([&] { gpuGroupBy(gS, gC); });

    // --- HYBRID: ---
    bool usedT = false, usedG = false;
    hybT[0] = timeIt([&] { usedT = planScan(T); gpuSelection(C, T); });
    double fracT = lastFrac;
    hybT[1] = timeIt([&] { hCnt = gpuCount(); });
    hybT[2] = timeIt([&] { hSum = gpuSum(); });
    hybT[3] = timeIt([&] { hMin = gpuMin(); });
    hybT[4] = timeIt([&] { hMax = gpuMax(); });
    usedG = planScan(TG);
    double fracG = lastFrac;
    gpuSelection(-1, TG);
    hybT[5] = timeIt([&] { gpuGroupBy(hS, hC); });


    bool ok[6];
    ok[1] = (cCnt == gCnt) && (cCnt == hCnt);
    ok[0] = ok[1];
    ok[2] = (cSum == gSum) && (cSum == hSum);
    ok[3] = (cMin == gMin) && (cMin == hMin);
    ok[4] = (cMax == gMax) && (cMax == hMax);
    ok[5] = true;
    for (int k = 0; k < NCAT; k++)
        ok[5] = ok[5] && cS[k] == gS[k] && cS[k] == hS[k] && cC[k] == gC[k] && cC[k] == hC[k];

    printf("\n=========== PART A: CPU vs CUDA vs HYBRID ===========\n");
    printf("Query: type = %s AND amount > %d   |   GROUP BY: amount > %d\n", typeNames[C], T, TG);
    printf("Answers: COUNT=%lld SUM=%lld MIN=%d MAX=%d\n\n", hCnt, hSum, hMin, hMax);
    printf("%-10s %9s %9s %10s %12s %6s\n", "Operator", "CPU ms", "CUDA ms", "Hybrid ms", "Hybrid/CPU", "Match");
    const char *names[6] = {"SELECTION", "COUNT", "SUM", "MIN", "MAX", "GROUP BY"};
    double cpuTot = 0, gpuTot = 0, hybTot = 0;
    for (int k = 0; k < 6; k++) {
        printf("%-10s %9.3f %9.3f %10.3f %11.1fx %6s\n", names[k], cpuT[k], gpuT[k], hybT[k],
               cpuT[k] / hybT[k], ok[k] ? "YES" : "NO");
        cpuTot += cpuT[k];  gpuTot += gpuT[k];  hybTot += hybT[k];
    }
    printf("%-10s %9.3f %9.3f %10.3f %11.1fx\n", "TOTAL", cpuTot, gpuTot, hybTot, cpuTot / hybTot);

    printf("\nOptimizer decisions (index used only if <= %.0f%% of rows qualify):\n", INDEX_LIMIT * 100);
    printf("  amount > %-6d: %4.1f%% of rows qualify -> %s\n", T,  100.0 * fracT, usedT ? "INDEX RANGE SCAN" : "FULL SCAN");
    printf("  amount > %-6d: %4.1f%% of rows qualify -> %s\n", TG, 100.0 * fracG, usedG ? "INDEX RANGE SCAN" : "FULL SCAN");

    printf("\nGROUP BY result (amount > %d):\n", TG);
    printf("%-10s %10s %18s\n", "Type", "COUNT", "SUM(amount)");
    for (int k = 0; k < NCAT; k++) printf("%-10s %10llu %18llu\n", typeNames[k], hC[k], hS[k]);

    // ================= PART B: GPU functions inside SQLite =================
    sqlite3_create_function(db, "gpu_agg",   3, SQLITE_UTF8, 0, gpuAggSQL,   0, 0);
    sqlite3_create_function(db, "gpu_group", 3, SQLITE_UTF8, 0, gpuGroupSQL, 0, 0);

    struct SqlTest { string name, nativeSQL, gpuSQL; };
    vector<SqlTest> tests = {
        {"SUM  (type = 4 AND amount > 200000)",
         "SELECT SUM(amount) FROM transactions WHERE type = 4 AND amount > 200000;",
         "SELECT gpu_agg('sum', 4, 200000);"},
        {"GROUP BY type, SUM (amount > 10000)",
         "SELECT type, SUM(amount) FROM transactions WHERE amount > 10000 GROUP BY type ORDER BY type;",
         "SELECT id, gpu_group('sum', id, 10000) FROM categories ORDER BY id;"}
    };


    vector<vector<pair<int, double>>> res(tests.size());
    vector<double> noIdx(tests.size()), withIdx(tests.size()), gpu(tests.size());
    for (size_t k = 0; k < tests.size(); k++)
        noIdx[k] = timeIt([&] { res[k] = runSQL(db, tests[k].nativeSQL); });


    sqlite3_exec(db, "CREATE INDEX idx_type_amount ON transactions(type, amount);", 0, 0, 0);

    printf("\n=========== PART B: SQLite vs GPU inside SQLite ===========\n");
    printf("%-38s %12s %12s %10s %6s\n", "Query", "No index ms", "Indexed ms", "GPU ms", "Match");
    for (size_t k = 0; k < tests.size(); k++) {
        vector<pair<int, double>> a, b;
        withIdx[k] = timeIt([&] { a = runSQL(db, tests[k].nativeSQL); });
        gpu[k]     = timeIt([&] { b = runSQL(db, tests[k].gpuSQL); });
        printf("%-38s %12.2f %12.2f %10.2f %6s\n", tests[k].name.c_str(), noIdx[k], withIdx[k], gpu[k],
               (sameRows(res[k], a) && sameRows(a, b)) ? "YES" : "NO");
    }

    printf("\nOne-time costs: read table from SQLite %.0f ms | copy to GPU %.0f ms | sorted index %.0f ms\n",
           loadTime, copyTime, indexTime);

    sqlite3_close(db);
    cudaFree(d_cat); cudaFree(d_amt); cudaFree(d_scat); cudaFree(d_samt);
    cudaFree(d_flag); cudaFree(d_pInt); cudaFree(d_pLL); cudaFree(d_gSum); cudaFree(d_gCnt);
    return 0;
}

Overwriting db_ops.cu


In [ ]:
!nvcc -std=c++14 db_ops.cu -o db_ops -lsqlite3
!./db_ops paysim.csv

PaySim loaded into SQLite: 6362620 transactions

=========== PART A: CPU vs CUDA vs HYBRID ===========
Query: type = TRANSFER AND amount > 200000   |   GROUP BY: amount > 10000
Answers: COUNT=409110 SUM=472928540529 MIN=200002 MAX=92445517

Operator      CPU ms   CUDA ms  Hybrid ms   Hybrid/CPU  Match
SELECTION     65.580     0.356      0.108       609.5x    YES
COUNT         33.848     0.780      0.227       149.0x    YES
SUM           35.813     0.854      0.256       140.0x    YES
MIN           44.654     0.717      0.226       197.8x    YES
MAX           42.101     0.764      0.253       166.1x    YES
GROUP BY     117.432     1.876      1.951        60.2x    YES
TOTAL        339.427     5.347      3.021       112.4x

Optimizer decisions (index used only if <= 50% of rows qualify):
  amount > 200000: 26.3% of rows qualify -> INDEX RANGE SCAN
  amount > 10000 : 79.8% of rows qualify -> FULL SCAN

GROUP BY result (amount > 10000):
Type            COUNT        SUM(amount)
CASH_IN      